# Tutorial: Estimating Social Contact Matrices with Prem Model

In [ ]:
from jax.random import PRNGKey

from cntmosaic.utils import AgeBins
from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import (
    ParticipantGenerator,
    MatrixGenerator,
    ContactGenerator,
    Stratification,
    PopulationConstructor,
)
from cntmosaic.vis import plot_mosaic, plot_mosaic_pixilated
from cntmosaic.dataloader.containers import PopulationData

from cntmosaic.models import Prem
from cntmosaic.analysis import ModelSummariserPrem, ModelEvaluatorPrem

from cntmosaic.utils import pixilate

import altair as alt

## 1. Introduction
The `Prem` class in `cntmosaic` is a Python implementation of the model put forth by Prem et al. (2017) for estimating social contact matrices from survey data.
This tutorial will guide you through how to use Prem's model and it's associated utilities within `cntmosaic` to analyze contact survey data.  

## 2. Synthetic Data Generation

We begin by generating some synthetic contact data for which we know the ground truth. To do so, we use `cntmosaic`'s data generation tools housed in the `sim` module to create realistic synthetic contact data.

We first load the contact pattern templates and population data:

In [ ]:
templates = load_template_patterns(country="United_States")
df_age_dist = load_age_distribution(country="United_States")

age_dist = df_age_dist.P.values

Define the population structure using `Stratification` and `PopulationConstructor`:

For an unstratified population, we create a single stratification with one level:

In [ ]:
# Create an unstratified population
unstratified = Stratification(
    name="general",
    n_strata=1,  # Single stratum = unstratified
    ref_age_dist=age_dist,
    labels=["All"],
    seed=42,
)

# Build population constructor
popcon = PopulationConstructor(unstratified)

**Key Concepts:**
- **Stratification**: Defines population subgroups (e.g., by gender, region, SES) with distinct age distributions
- **PopulationConstructor**: Combines one or more stratifications to build the joint population structure
- For unstratified analysis (as in this tutorial), use a single stratification with `n_strata=1`

The `Stratification` class uses Gaussian Process-based perturbations to generate realistic age distributions for each stratum.

We randomly simulate a contact intensity matrix using the `MatrixGenerator` class.

In [ ]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
# For unstratified, use generate_single() with mean_intensity parameter
contact_matrix = matrix_gen.generate_single(
    popcon=popcon,
    mean_intensity=10.0,  # Average 10 contacts per person
    seed=42,  # For reproducibility
)

print("Matrix shape:", contact_matrix["All->All"].shape)
print("Average contacts per person:", contact_matrix["All->All"].sum(axis=0).mean())

Visualize the generated contact matrix with `plot_mosaic` function:

In [ ]:
chart_fine = plot_mosaic(
    contact_matrix["All->All"],
    title="Synthetic Contact Intensity Matrix",
    zlabel="intensity",
)

age_bins = AgeBins(0, 80, 5)
contact_matrix_coarse = pixilate(
    contact_matrix["All->All"], age_bins, df_age_dist["P"].values
)
chart_coarse = plot_mosaic_pixilated(
    contact_matrix_coarse,
    age_bins=age_bins,
    title="Pixilated Contact Intensity Matrix",
    zlabel="intensity",
)

alt.hconcat(chart_fine, chart_coarse)

The `ParticipantGenerator` class simulates participant data based on the population structure:

In [ ]:
# Initialize the participant generator
part_gen = ParticipantGenerator(popcon, n_part=1000)

# Generate participants
df_part = part_gen.generate(seed=42)

print(df_part.head())

The `ContactGenerator` class simulates contact data for the participants:

In [ ]:
cnt_gen = ContactGenerator(
    df_part=df_part, cint_matrices=contact_matrix, model="poisson"
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

Plot the empirical contact matrix from the simulated contact data:

In [ ]:
emp_cint_matrix = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)
plot_mosaic(
    emp_cint_matrix["All->All"], title="Empirical Contact Matrix", zlabel="Intensity"
)

## 3. Estimating Contact Matrices with the Prem Model

### 3.1 Data Preparation

The Prem model requires:
1. Age bins for grouping participants and contacts (`AgeBins` object)
2. A column `age_part` in the participant dataframe

Here, we define 5-year age bins from 0 to 80 years:

In [ ]:
from cntmosaic.dataloader import ParticipantData, ContactData

part_data = ParticipantData(df_part, id_col="id", age_col="age")
cnt_data = ContactData(df_cnt, id_col="id", age_col="age_cnt")
age_bins = AgeBins(0, 80, 5)

### 3.2 Basic Matrix Estimation

In [ ]:
# Initialize Prem Model
prem = Prem(part_data, cnt_data, age_bins=age_bins)

#### Fast Inference using SVI
For quick estimation of contact matrices, we can use Stochastic Variational Inference (SVI) via the `run_inference_svi` method.

In [ ]:
prem.run_inference_svi(PRNGKey(1), num_steps=5000)

In [ ]:
# Create PopulationData instance for model summarization
# PopulationData contains fine-grained age distribution for depixilation
pop_data = PopulationData(df_age_dist, age_col="age", size_col="P")

# Initialize summariser with PopulationData
summariser = ModelSummariserPrem(prem, pop_data=pop_data)

In [ ]:
# Summarise the posterior contact intensity matrix
# For K=1 (unstratified), output is NDArray of shape (3, A, A)
# where dimension 0 contains [lower, median, upper] quantiles
summary = summariser.summarise_cint(alpha=0.05)

# Extract quantiles from the array
cint_lower = summary[0]  # 2.5th percentile
cint_median = summary[1]  # 50th percentile (median)
cint_upper = summary[2]  # 97.5th percentile

# Plot the inferred contact intensity matrices
plt_median = plot_mosaic_pixilated(
    cint_median,
    age_bins,
    title="Inferred Contact Intensity Matrix (Median)",
    zlabel="Intensity",
    ylabel=None,
)
plt_lower = plot_mosaic_pixilated(
    cint_lower,
    age_bins,
    title="Inferred Contact Intensity Matrix (2.5th Percentile)",
    zlabel="Intensity",
)
plt_upper = plot_mosaic_pixilated(
    cint_upper,
    age_bins,
    title="Inferred Contact Intensity Matrix (97.5th Percentile)",
    zlabel="Intensity",
    ylabel=None,
)

alt.hconcat(plt_lower, plt_median, plt_upper)

### Applying Reciprocity Adjustment

The Prem model can apply reciprocity adjustment to ensure balanced contact flows weighted by population. This enforces the constraint that the total contacts from group A to group B should equal the total contacts from B to A when weighted by population sizes.

In [ ]:
# Apply reciprocity adjustment to the contact intensity matrix
summary_recip = summariser.summarise_cint(alpha=0.05, apply_reciprocity=True)
cint_median_recip = summary_recip[1]  # Extract median

plt_recip = plot_mosaic_pixilated(
    cint_median_recip,
    age_bins,
    title="Reciprocity-Adjusted Contact Intensity (Median)",
    zlabel="Intensity",
    ylabel=None,
)

alt.hconcat(plt_median, plt_recip)

### Model Evaluation

We can evaluate the estimated contact matrix against the true contact matrix using the `ModelEvaluatorPrem` class. Here, we include extended metrics in the evaluation.

In [ ]:
evaluator = ModelEvaluatorPrem(summariser, contact_matrix["All->All"])

In [ ]:
evaluator.evaluate()

In [ ]:
# Create an unstratified population
strat = Stratification(
    name="sex",
    n_strata=2,  # Single stratum = unstratified
    ref_age_dist=age_dist,
    labels=["M", "F"],
    seed=42,
)

# Build population constructor
popcon = PopulationConstructor(strat)

In [ ]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
# For unstratified, use generate_single() with mean_intensity parameter
contact_matrix = matrix_gen.generate_partial(
    popcon=popcon,
    mean_intensity=15.0,  # Average 15 contacts per person
    seed=42,  # For reproducibility
)

print("Number of matrices:", len(contact_matrix.keys()))
print("Keys:", contact_matrix.keys())
print("Matrix shape:", contact_matrix["M->All"].shape)
print("Average contacts per person:", contact_matrix["M->All"].sum(axis=0).mean())

In [ ]:
# Initialize the participant generator
part_gen = ParticipantGenerator(popcon, n_part=2000)

# Generate participants
df_part = part_gen.generate(seed=42)

print(df_part.head())

In [ ]:
cnt_gen = ContactGenerator(
    df_part=df_part, cint_matrices=contact_matrix, model="poisson"
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

In [ ]:
part_data = ParticipantData(df_part, id_col="id", age_col="age", strat_var_cols="sex")
cnt_data = ContactData(df_cnt, id_col="id", age_col="age_cnt")
age_bins = AgeBins(0, 80, 5)

In [ ]:
model_partial = Prem(part_data, cnt_data, age_bins)
model_partial.print_model_shape()

In [ ]:
rng_key = PRNGKey(0)
model_partial.run_inference_svi(rng_key, num_steps=5000)

In [ ]:
# Create PopulationData instance for model summarization
df_pop = popcon.df_P

# PopulationData contains fine-grained age distribution for depixilation
pop_data = PopulationData(df_pop, age_col="age", size_col="P", strat_var_cols="sex")

# Initialize summariser with PopulationData
summariser = ModelSummariserPrem(model_partial, pop_data=pop_data)

In [ ]:
# Test summarisation with stratified model
# For K=2 (sex stratification), output is Dict[str, NDArray]
# where keys are stratum labels and values are (3, A, A) arrays
summary_partial = summariser.summarise_cint(alpha=0.05)

print(f"Summary type: {type(summary_partial)}")
print(f"Keys: {list(summary_partial.keys())}")
print(f"Shape for 'M': {summary_partial['M->All'].shape}")

In [ ]:
chart_m_all = plot_mosaic_pixilated(
    summary_partial["M->All"][1],
    age_bins,
    title="Inferred Contact Intensity Matrix for Men (Median)",
    zlabel="Intensity",
    ylabel=None,
)

chart_f_all = plot_mosaic_pixilated(
    summary_partial["F->All"][1],
    age_bins,
    title="Inferred Contact Intensity Matrix for Women (Median)",
    zlabel="Intensity",
    ylabel=None,
)

alt.hconcat(chart_m_all, chart_f_all)

In [ ]:
evaluator = ModelEvaluatorPrem(summariser, contact_matrix)

In [ ]:
evaluator.evaluate()

In [ ]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
contact_matrix = matrix_gen.generate_full(
    popcon=popcon,
    mean_intensity=10.0,  # Average 10 contacts per person
    seed=42,  # For reproducibility
)

In [ ]:
# Initialize the participant generator
part_gen = ParticipantGenerator(popcon, n_part=1000)

# Generate participants
df_part = part_gen.generate(seed=42)

print(df_part.head())

In [ ]:
cnt_gen = ContactGenerator(
    df_part=df_part, cint_matrices=contact_matrix, model="poisson"
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

In [ ]:
part_data = ParticipantData(df_part, id_col="id", age_col="age", strat_var_cols="sex")
cnt_data = ContactData(df_cnt, id_col="id", age_col="age_cnt", strat_var_cols="sex_cnt")
age_bins = AgeBins(0, 80, 5)  # Changed from 10 to 5 to match partial model

In [ ]:
model_full = Prem(part_data, cnt_data, age_bins)
model_full.print_model_shape()

In [ ]:
rng_key = PRNGKey(0)
model_full.run_inference_svi(rng_key, num_steps=5000)

In [ ]:
# Initialize summariser with PopulationData
summariser = ModelSummariserPrem(model_full, pop_data=pop_data)

In [ ]:
# Test summarisation with stratified model
summary_full = summariser.summarise_cint(alpha=0.05, apply_reciprocity=True)

print(f"Summary type: {type(summary_full)}")
print(f"Keys: {list(summary_full.keys())}")
print(f"Shape for 'M': {summary_full['M->M'].shape}")

In [ ]:
evaluator = ModelEvaluatorPrem(summariser, contact_matrix)

In [ ]:
evaluator.evaluate()